In [1]:
import pandas as pd
import sqlite3

# 连接数据库 (模拟 Hive 数仓)
conn = sqlite3.connect('tmall_dw.db')
cursor = conn.cursor()

# ==========================================
# 1. ODS 层 (Operational Data Store) - 原始数据层
# 目标：保持数据原貌，不做任何修改
# ==========================================
print(">>> [ODS层] 正在加载原始数据...")

# 模拟从业务系统抽取数据 (Sqoop/DataX)
df_raw = pd.read_csv('tmall_order_report.csv') 
# 简单修复列名空格，但这属于加载前的预处理
df_raw.columns = df_raw.columns.str.strip() 

# 存入 ODS 表
df_raw.to_sql('ods_tmall_order', conn, if_exists='replace', index=False)
print(">>> ODS 层加载完成: ods_tmall_order (全量覆盖)")


# ==========================================
# 2. DWD 层 (Data Warehouse Detail) - 明细层
# 目标：清洗、规范化、维度建模 (星型模型)
# ==========================================
print("\n>>> [DWD层] 正在进行维度建模 (星型模型)...")

# 读取 ODS 数据
df_ods = pd.read_sql("SELECT * FROM ods_tmall_order", conn)

# --- Step 2.1: 数据清洗 (ETL) ---
# 转换时间格式
df_ods['create_time'] = pd.to_datetime(df_ods['订单创建时间'])
df_ods['pay_time'] = pd.to_datetime(df_ods['订单付款时间'])
# 提取日期 (用于分区)
df_ods['dt'] = df_ods['create_time'].dt.strftime('%Y-%m-%d')
# 处理空值 (假设订单没付款，实付金额填0)
df_ods['买家实际支付金额'] = df_ods['买家实际支付金额'].fillna(0.0)

# --- Step 2.2: 维度拆分 (Dimension Tables) ---
# [维度表 1]: 省份维度表 (dim_province)
# 只要唯一的省份列表
dim_province = df_ods[['收货地址']].drop_duplicates().reset_index(drop=True)
dim_province.columns = ['province_name']
dim_province['province_id'] = dim_province.index + 100 # 生成一个 ID
dim_province.to_sql('dim_province', conn, if_exists='replace', index=False)

# --- Step 2.3: 事实表构建 (Fact Table) ---
# [事实表]: 订单事实表 (dwd_fact_order)
# 将省份名称替换为 ID (关联维度表)
dwd_fact = pd.merge(df_ods, dim_province, left_on='收货地址', right_on='province_name', how='left')

# 只保留核心业务字段 + 外键 ID
dwd_fact_order = dwd_fact[[
    '订单编号', 
    'province_id',    # 外键 -> 关联 dim_province
    '总金额', 
    '买家实际支付金额', 
    '退款金额',
    'create_time', 
    'pay_time',
    'dt'
]]
dwd_fact_order.columns = ['order_id', 'province_id', 'total_amount', 'actual_pay', 'refund_amount', 'create_time', 'pay_time', 'dt']

dwd_fact_order.to_sql('dwd_fact_order', conn, if_exists='replace', index=False)
print(">>> DWD 层构建完成: 事实表(dwd_fact_order) + 维度表(dim_province)")


# ==========================================
# 3. DWS 层 (Data Warehouse Service) - 汇总层
# 目标：按主题聚合，减少数据量，加快查询
# ==========================================
print("\n>>> [DWS层] 正在生成每日汇总表...")

# 不需要每次都查明细，按“天”和“省份”汇总
sql_dws = """
SELECT 
    dt,
    province_id,
    COUNT(order_id) as order_count,
    SUM(total_amount) as gmv,
    SUM(actual_pay) as payment_amt,
    SUM(refund_amount) as refund_amt
FROM dwd_fact_order
GROUP BY dt, province_id
"""
dws_daily = pd.read_sql(sql_dws, conn)
dws_daily.to_sql('dws_daily_province_sales', conn, if_exists='replace', index=False)
print(">>> DWS 层构建完成: dws_daily_province_sales (轻度汇总)")


# ==========================================
# 4. ADS 层 (Application Data Service) - 应用层
# 目标：给报表和老板看的最终指标
# ==========================================
print("\n>>> [ADS层] 正在生成最终报表...")

# 指标 1: 各省份 GMV 排名 (关联维度表获取省份名)
sql_ads_rank = """
SELECT 
    p.province_name,
    SUM(s.gmv) as total_gmv,
    SUM(s.payment_amt) as total_actual_pay
FROM dws_daily_province_sales s
JOIN dim_province p ON s.province_id = p.province_id
GROUP BY p.province_name
ORDER BY total_gmv DESC
LIMIT 5
"""
ads_province_rank = pd.read_sql(sql_ads_rank, conn)

print("--- [ADS 报表] 省份销售 Top 5 ---")
print(ads_province_rank)

conn.close()

>>> [ODS层] 正在加载原始数据...
>>> ODS 层加载完成: ods_tmall_order (全量覆盖)

>>> [DWD层] 正在进行维度建模 (星型模型)...
>>> DWD 层构建完成: 事实表(dwd_fact_order) + 维度表(dim_province)

>>> [DWS层] 正在生成每日汇总表...
>>> DWS 层构建完成: dws_daily_province_sales (轻度汇总)

>>> [ADS层] 正在生成最终报表...
--- [ADS 报表] 省份销售 Top 5 ---
  province_name  total_gmv  total_actual_pay
0            上海  544907.63         264039.78
1            北京  231055.49         166448.48
2           江苏省  227930.93         159359.18
3           广东省  227855.28         147822.90
4           浙江省  203126.96         141664.80
